In [ ]:
# Run the configured point-prediction script with this notebook R kernel.
# Leave empty for automatic lookup, or set the clone root, codebase/v2, or codebase/v2/src.
project_path <- Sys.getenv("SEAGRASS_V2_ROOT", "")

kernel_cwd <- getwd()
cat("Kernel working directory:", kernel_cwd, "\n")
if (!is.character(project_path) || length(project_path) != 1L || is.na(project_path)) {
  stop("project_path must be a single directory path.", call. = FALSE)
}
start_dirs <- if (nzchar(project_path)) path.expand(project_path) else {
  c(kernel_cwd, file.path(kernel_cwd, "Seagrass_carbon_stock_model"))
}
offsets <- c("", "src", file.path("codebase", "v2", "src"))
checked <- character()
bootstrap_path <- NA_character_
for (start in start_dirs) {
  directory <- normalizePath(start, winslash = "/", mustWork = FALSE)
  for (i in seq_len(40L)) {
    for (offset in offsets) {
      candidate <- file.path(directory, offset)
      checked <- c(checked, candidate)
      markers <- file.path(candidate, c("predict_gpr_at_points.R",
        file.path("modelling", "R", "init_repo.R"),
        file.path("modelling", "R", "project_root.R")))
      if (all(file.exists(markers))) {
        bootstrap_path <- markers[[3L]]
        break
      }
    }
    if (!is.na(bootstrap_path) || nzchar(project_path)) break
    parent <- dirname(directory)
    if (identical(parent, directory)) break
    directory <- parent
  }
  if (!is.na(bootstrap_path)) break
}
if (is.na(bootstrap_path)) {
  stop(if (nzchar(project_path)) paste0("Invalid project_path: ", project_path) else
    "Cannot find the v2 project root.",
    "\nKernel working directory: ", kernel_cwd,
    "\nSet project_path in this cell (or SEAGRASS_V2_ROOT) to the clone root, codebase/v2, or codebase/v2/src.",
    "\nChecked:\n  ", paste(unique(checked), collapse = "\n  "), call. = FALSE)
}

sys.source(bootstrap_path, envir = .GlobalEnv)
project_root <- seagrass_find_v2_root(kernel_cwd, project_path = project_path)
script_path <- file.path(project_root, "predict_gpr_at_points.R")
cat("v2 project root:", project_root, "\n")
local({
  previous_root <- Sys.getenv("SEAGRASS_V2_ROOT", unset = NA_character_)
  Sys.setenv(SEAGRASS_V2_ROOT = project_root)
  tryCatch(source(script_path, local = .GlobalEnv), finally = {
    if (is.na(previous_root)) Sys.unsetenv("SEAGRASS_V2_ROOT") else
      Sys.setenv(SEAGRASS_V2_ROOT = previous_root)
  })
})


In [ ]:
# Startup diagnostics after running the first cell.
if (!exists("seagrass_find_v2_root", mode = "function")) {
  stop("Run the first cell to initialize project discovery.", call. = FALSE)
}
print(kernel_cwd)
print(script_path)
print(seagrass_find_v2_root(kernel_cwd, project_path = project_path))
print(getwd())
